# Setup

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path
from hydra.utils import instantiate
import torch
from omegaconf import OmegaConf
from src.utils.notebook_setup import init_nlp_notebook

# 1. Вычисляем корень проекта ОДИН РАЗ
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# 2. Регистрируем все резолверы ГЛОБАЛЬНО
# Теперь ${paths.data_dir} будет подтягиваться из конфига, 
# а если Hydra падает, мы подставляем ${PROJECT_ROOT}
OmegaConf.register_new_resolver("project_root", lambda: str(PROJECT_ROOT))
OmegaConf.register_new_resolver("hydra", lambda path: str(PROJECT_ROOT), replace=True)
OmegaConf.register_new_resolver("now", lambda *args: "now", replace=True)

# 3. Инициализируем Hydra
cfg = init_nlp_notebook()

# 4. Трюк: принудительно подменяем переменную paths, если она не разрешилась
# Это поможет конфигу увидеть, где лежат данные, без правок yaml-файлов
if "paths" not in cfg:
    cfg.paths = OmegaConf.create()
cfg.paths.data_dir = str(PROJECT_ROOT / "data")

device = "cuda" if torch.cuda.is_available() else "cpu"

NLP Environment ready. Root: c:\fake-news-detection-ml-system


# Data init and collator

In [2]:
tokenizer = instantiate(cfg.model.tokenizer).build()
from src.core.data.builder import NLPDataModule

datamodule = NLPDataModule(
    data_cfg=cfg.data, 
    tokenizer=tokenizer
)

datamodule.prepare_data()
datamodule.setup(stage="validate")
val_dataloader = datamodule.val_dataloader()
sample_batch = next(iter(val_dataloader))

print("Keys in batch:", sample_batch.keys())
print("Input IDs shape:", sample_batch["input_ids"].shape)
if "labels" in sample_batch:
    print("Labels shape:", sample_batch["labels"].shape)

c:\fake-news-detection-ml-system\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:src.core.models.tokenization:Загрузка токенизатора: DeepPavlov/rubert-base-cased
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased/4036cab694767a299f2b9e6492909664d9414229/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased/4036cab694767a299f2b9e6492909664d9414229/tokenizer_config.json "HTTP/1.1 20

Keys in batch: KeysView({'input_ids': tensor([[   101,  11253,    269,  ...,  15000,  10636,    102],
        [   101,  12032,  12463,  ...,      0,      0,      0],
        [   101,    308,    269,  ...,      0,      0,      0],
        ...,
        [   101,  33162,   7046,  ...,      0,      0,      0],
        [   101, 110657,  13748,  ...,      0,      0,      0],
        [   101,  43364,   7491,  ...,      0,      0,      0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0])})
Input IDs shape: tor

# Model init

In [ ]:
base_model = instantiate(cfg.model.builder, tokenizer=tokenizer).build()
model = instantiate(cfg.model_module, model=base_model)
model.to(device)
model.eval()
print(f"Model architecture: {model.__class__.__name__}")

INFO:src.core.models.builder:Загрузка базовой модели: DeepPavlov/rubert-base-cased
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased/4036cab694767a299f2b9e6492909664d9414229/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased/4036cab694767a299f2b9e6492909664d9414229/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22113.89it/s]
[transformers] BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased

Model architecture: NLPModel


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/DeepPavlov/rubert-base-cased "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/DeepPavlov/rubert-base-cased/commits/main "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/DeepPavlov/rubert-base-cased/discussions?p=0 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/DeepPavlov/rubert-base-cased/commits/refs%2Fpr%2F4 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/refs%2Fpr%2F4/model.safetensors.index.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/refs%2Fpr%2F4/model.safetensors "HTTP/1.1 302 Found"


# Forward Pass

In [24]:
sample_batch = {k: v.to(device) for k, v in sample_batch.items() if isinstance(v, torch.Tensor)}

with torch.no_grad():
    # Проверка кастомного forward()
    outputs = model(
        input_ids=sample_batch["input_ids"], 
        attention_mask=sample_batch["attention_mask"]
    )

print("Output keys:", outputs.keys())
print("Sentiment logits shape:", outputs["last_hidden_state"].shape)
print("Category logits shape:", outputs["pooler_output"].shape)

# Проверка инициализации твоего LightningModule (NLPModel)
lightning_module = instantiate(
    cfg.model_module,
    model=model,
    num_classes=2 # Передаем аргументы, которые ожидаются в конструкторе NLPModel
)
print(f"Metrics initialized: {lightning_module.val_metrics}")

Output keys: odict_keys(['last_hidden_state', 'pooler_output'])
Sentiment logits shape: torch.Size([16, 512, 768])
Category logits shape: torch.Size([16, 768])
Metrics initialized: MetricCollection(
  (acc): MulticlassAccuracy()
  (f1): MulticlassF1Score(),
  prefix=val_
)


# Baseline Evaluation

In [26]:
from sklearn.metrics import classification_report
from tqdm.auto import tqdm
import numpy as np

all_preds = []
all_labels = []

# Ограничимся 50 батчами для быстрой проверки
max_batches = 50 

for i, batch in enumerate(tqdm(val_dataloader, desc="Baseline Eval", total=min(len(val_dataloader), max_batches))):
    if i >= max_batches:
        break
        
    batch = {k: v.to(device) for k, v in batch.items()}
    
    with torch.no_grad():
        outputs = model(**batch)
        
    preds = torch.argmax(outputs["pooler_output"], dim=-1)
    
    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(batch["labels"].cpu().numpy())

print("\n--- UNTRAINED BASELINE REPORT ---")
print(classification_report(all_labels, all_preds, zero_division=0))

# Ожидание: Accuracy должно быть около (1 / количество_классов).
# Если Accuracy сильно выше случайного угадывания — возможно, произошла утечка данных (Data Leakage).

Baseline Eval: 100%|██████████| 50/50 [07:08<00:00,  8.58s/it]



--- UNTRAINED BASELINE REPORT ---
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     603.0
           1       0.00      0.00      0.00     197.0
         115       0.00      0.00      0.00       0.0
         148       0.00      0.00      0.00       0.0
         323       0.00      0.00      0.00       0.0
         442       0.00      0.00      0.00       0.0
         494       0.00      0.00      0.00       0.0
         525       0.00      0.00      0.00       0.0
         615       0.00      0.00      0.00       0.0
         624       0.00      0.00      0.00       0.0
         761       0.00      0.00      0.00       0.0

    accuracy                           0.00     800.0
   macro avg       0.00      0.00      0.00     800.0
weighted avg       0.00      0.00      0.00     800.0

